In [1]:
# IMPORT PACKAGES

from pathlib import Path

import numpy as np
import rasterio

from tqdm import tqdm

In [2]:
# ROOT DIRECTORY

ROOT_DIR = Path("..")

# INPUT DATASET
INPUT_DIR = ROOT_DIR / "dataset_aligned"

# OUTPUT DIRECTORY
OUTPUT_DIR = ROOT_DIR / "indices"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Paths ready.")


Paths ready.


In [3]:
# FIND YEARS

year_folders = sorted([
    f for f in INPUT_DIR.iterdir()
    if f.is_dir()
])

print(year_folders)

[WindowsPath('dataset_aligned/2025'), WindowsPath('dataset_aligned/2026')]


In [4]:
def compute_index(band_1, band_2):

    band_1 = band_1.astype("float32")
    band_2 = band_2.astype("float32")

    np.seterr(divide='ignore', invalid='ignore')

    index = (band_1 - band_2) / (band_1 + band_2)

    index[np.isinf(index)] = np.nan

    return index

In [5]:
def compute_indices():

    for year_folder in year_folders:

        year_name = year_folder.name

        print(f"\nPROCESSING YEAR: {year_name}")

        # OUTPUT YEAR
        output_year = OUTPUT_DIR / year_name
        output_year.mkdir(exist_ok=True)

        # FIND FLIGHTS
        flight_folders = sorted([
            f for f in year_folder.iterdir()
            if f.is_dir()
        ])

        for flight_folder in tqdm(flight_folders):

            flight_name = flight_folder.name

            print(f"\nPROCESSING FLIGHT: {flight_name}")

            # OUTPUT FLIGHT
            output_flight = output_year / flight_name
            output_flight.mkdir(exist_ok=True)

            # -----------------------------------------
            # CHECK IF ALREADY PROCESSED
            # -----------------------------------------

            expected_output = (
                output_flight / "NDVI.tif"
            )

            if expected_output.exists():

                print("Flight already processed. Skipping.")

                continue

            # FIND TIFFS
            tif_files = list(flight_folder.glob("*.tif"))

            # -----------------------------------------
            # FIND REQUIRED BANDS
            # -----------------------------------------

            red_path = None
            green_path = None
            nir_path = None
            rededge_path = None
            thermal_path = None

            for tif in tif_files:

                name = tif.name.lower()

                if "red_reflectance" in name:
                    red_path = tif

                elif "green_reflectance" in name:
                    green_path = tif

                elif "nir_reflectance" in name:
                    nir_path = tif

                elif "rededge_reflectance" in name:
                    rededge_path = tif

                elif "thermal" in name:
                    thermal_path = tif

            # -----------------------------------------
            # CHECK REQUIRED BANDS
            # -----------------------------------------

            if not all([
                red_path,
                green_path,
                nir_path,
                rededge_path
            ]):

                print("Missing required bands.")
                continue

            # -----------------------------------------
            # READ RED
            # -----------------------------------------

            with rasterio.open(red_path) as src:

                red = src.read(1).astype("float32")

                profile = src.profile

            # -----------------------------------------
            # READ GREEN
            # -----------------------------------------

            with rasterio.open(green_path) as src:

                green = src.read(1).astype("float32")

            # -----------------------------------------
            # READ NIR
            # -----------------------------------------

            with rasterio.open(nir_path) as src:

                nir = src.read(1).astype("float32")

            # -----------------------------------------
            # READ RED EDGE
            # -----------------------------------------

            with rasterio.open(rededge_path) as src:

                rededge = src.read(1).astype("float32")

            # -----------------------------------------
            # COMPUTE INDICES
            # -----------------------------------------

            ndvi = compute_index(nir, red)

            ndre = compute_index(nir, rededge)

            gndvi = compute_index(nir, green)

            # -----------------------------------------
            # UPDATE PROFILE (in order to update the metadata)
            # -----------------------------------------

            profile.update(
                dtype=rasterio.float32,
                count=1,
                compress="lzw",
                nodata=np.nan
            )

            # -----------------------------------------
            # SAVE NDVI
            # -----------------------------------------

            ndvi_output = output_flight / "NDVI.tif"

            with rasterio.open(     #for creating a new raster
                ndvi_output,
                "w",
                **profile
            ) as dst:

                dst.write(
                    ndvi.astype(rasterio.float32),
                    1
                )

            print("Saved NDVI")

            # -----------------------------------------
            # SAVE NDRE
            # -----------------------------------------

            ndre_output = output_flight / "NDRE.tif"

            with rasterio.open(
                ndre_output,
                "w",
                **profile
            ) as dst:

                dst.write(
                    ndre.astype(rasterio.float32),
                    1
                )

            print("Saved NDRE")

            # -----------------------------------------
            # SAVE GNDVI
            # -----------------------------------------

            gndvi_output = output_flight / "GNDVI.tif"

            with rasterio.open(
                gndvi_output,
                "w",
                **profile
            ) as dst:

                dst.write(
                    gndvi.astype(rasterio.float32),
                    1
                )

            print("Saved GNDVI")

            # -----------------------------------------
            # SAVE THERMAL
            # -----------------------------------------

            if thermal_path is not None:

                thermal_output = (
                    output_flight /
                    "canopy_temperature.tif"
                )

                with rasterio.open(thermal_path) as src:

                    thermal = src.read(1)

                    thermal_profile = src.profile

                with rasterio.open(
                    thermal_output,
                    "w",
                    **thermal_profile
                ) as dst:

                    dst.write(thermal, 1)

                print("Saved canopy temperature")

    print("\nALL INDICES COMPUTED.")

In [6]:
compute_indices()


PROCESSING YEAR: 2025


100%|█████████████████████████████████████████████████████████████████████████████████| 11/11 [00:00<00:00, 563.32it/s]



PROCESSING FLIGHT: flight_2025_04_09
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_05_08
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_05_14
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_05_19
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_06_03
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_06_12
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_06_19
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_07_04
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_07_08
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_08_11
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2025_08_19
Flight already processed. Skipping.

PROCESSING YEAR: 2026


  0%|                                                                                            | 0/9 [00:00<?, ?it/s]


PROCESSING FLIGHT: flight_2026_04_01
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2026_04_07
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2026_04_15
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2026_04_22
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2026_05_01
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2026_05_22
Flight already processed. Skipping.

PROCESSING FLIGHT: flight_2026_05_28
Saved NDVI
Saved NDRE


...

Saved GNDVI
Saved canopy temperature

PROCESSING FLIGHT: flight_2026_06_01
Saved NDVI
Saved NDRE


 89%|██████████████████████████████████████████████████████████████████████████▋         | 8/9 [00:07<00:01,  1.11s/it]

Saved GNDVI
Saved canopy temperature

PROCESSING FLIGHT: flight_2026_06_19
Saved NDVI
Saved NDRE


100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:11<00:00,  1.29s/it]

Saved GNDVI
Saved canopy temperature

ALL INDICES COMPUTED.
